In [521]:
import pandas as pd
import math
import numpy as np

In [522]:
df = pd.read_csv("nba_2008-2025.csv")
df

,season,date,regular,playoffs,away,home,score_away,score_home,q1_away,q2_away,...,ot_home,whos_favored,spread,total,moneyline_away,moneyline_home,h2_spread,h2_total,id_spread,id_total
0,2008,10/30/2007,True,False,por,sas,97,106,26,23,...,0,home,13.0,189.5,900.0,-1400.0,5.0,95.0,0.0,1
1,2008,10/30/2007,True,False,uta,gsw,117,96,28,34,...,0,home,1.0,212.0,100.0,-120.0,3.0,105.5,0.0,1
2,2008,10/30/2007,True,False,hou,lal,95,93,16,27,...,0,away,5.0,199.0,-230.0,190.0,3.0,99.0,0.0,0
3,2008,10/31/2007,True,False,phi,tor,97,106,22,28,...,0,home,6.5,191.0,255.0,-305.0,2.0,96.5,1.0,1
4,2008,10/31/2007,True,False,was,ind,110,119,23,22,...,16,away,1.5,203.5,-125.0,105.0,1.0,105.0,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23113,2025,6/11/2025,False,True,okc,ind,107,116,32,28,...,0,away,4.5,225.5,NaN,NaN,NaN,NaN,0.0,0
23114,2025,6/13/2025,False,True,okc,ind,111,104,34,23,...,0,away,6.5,227.5,NaN,NaN,NaN,NaN,1.0,0
23115,2025,6/16/2025,False,True,ind,okc,109,120,22,23,...,0,home,8.5,223.5,NaN,NaN,NaN,NaN,1.0,1
23116,2025,6/19/2025,False,True,okc,ind,91,108,25,17,...,0,away,5.5,222.5,NaN,NaN,NaN,NaN,0.0,0


In [523]:
class Basketball_Elo:
    def __init__(self, elo_start = 1500, K = 20, H = 60, starting_cash = 1000):
        self.teams = ['SAS', 'GSW', 'LAL', 'TOR', 'IND', 'ORL', 'BKN', 'CLE', 'MEM', 'NOP', 'DEN', 'MIA', 'UTA', 'OKC', 'CHA', 'ATL', 'BOS', 'MIN', 'CHI', 'PHX', 'LAC', 'PHI', 'WAS', 'MIL', 'HOU', 'DAL', 'NYK', 'DET', 'SAC', 'POR']
        self.start = elo_start
        self.start_hfa_elo = H
        self.start_cash = starting_cash
        self.elo = {}
        self.hfa_elo = {}
        self.cash = starting_cash
        self.K = K
    
    def reset(self):
        '''Reset and Intializes Elo rating '''
        self.cash = self.start_cash
        for x in self.teams:
            self.elo[x] = self.start
            self.hfa_elo[x] = self.start_hfa_elo
    
    def win_prob(self, home_abr, away_abr):
        ''''
        Probability of Winning for the home team when you provide the ratings and homefield advantage
        '''
        p_home = 1 / (1 + 10 ** -((self.elo[home_abr] + self.hfa_elo[home_abr] - self.elo[away_abr])/400))
        p_away = 1 - p_home
        return p_home, p_away
    
    def update_rating(self, home_abr, away_abr, home_win):
        ''''
        This updates the teams Elo Ratings
        '''
        p_home, p_away= self.win_prob(home_abr, away_abr)
        #updates the new rating
        self.elo[home_abr] += self.K * (home_win - p_home)
        self.elo[away_abr] += self.K * ((1 - home_win) - (p_away))

    def american_to_decimal(self,price):
        if price > 0:
            sigma = 1 + (price / 100) 
        else:
            sigma = 1 + (100/ - price)
        return sigma
    
    def expected_value(self,home_abr, away_abr, h_price, a_price):
        '''This gives you the value of a bet'''
        sigma_h = self.american_to_decimal(h_price)
        sigma_a = self.american_to_decimal(a_price)
        beta_h = sigma_h - 1
        beta_a = sigma_a - 1
        prob_h, prob_a = self.win_prob(home_abr, away_abr)
        ev_home = prob_h * beta_h - (1 - prob_h)
        ev_away = prob_a * beta_a - (1 - prob_a)
        return ev_home, ev_away , beta_h, beta_a
    
    def Kelly_fraction(self, home_abr, away_abr, h_price, a_price, max_bet = 0.05, kelly_limit = 2):
        '''
        This gives you how much you should bet
        max_bet should be in fraction default is 0.05
        '''
        ev_home, ev_away , beta_h, beta_a = self.expected_value(home_abr, away_abr, h_price, a_price) 
        f_h  = ev_home / beta_h
        f_a = ev_away / beta_a
        h_kelly = min(max(f_h,0),max_bet) / kelly_limit
        a_kelly =min(max(f_a,0),max_bet) / kelly_limit
        return h_kelly, a_kelly
    
    def cash_multi(self, odds):
        if odds > 0:
            return (odds / 100)
        else:
            return (100/ abs(odds))
    
    def simulation(self, home_abr, away_abr, h_price, a_price, h_win, max_bet = 0.05,  kelly_limit = 2):
        '''this is simulation'''
        h_kelly, a_kelly  = self.Kelly_fraction(home_abr, away_abr, h_price, a_price, max_bet, kelly_limit)

        # adds wins minus the loss if betting on both sides
        if h_win:
            profit = (h_kelly * self.cash * self.cash_multi(h_price)) - (a_kelly * self.cash)
            self.cash += profit
        else:
            profit = (a_kelly * self.cash * self.cash_multi(a_price)) - (h_kelly * self.cash)
            self.cash += profit
        return self.cash, profit

        


In [524]:
test = Basketball_Elo()
test.reset()

In [525]:
test.win_prob('MIA', 'UTA')

(0.5854986786718095, 0.4145013213281905)

In [526]:
test.update_rating('MIA', 'UTA', 1)

In [527]:
test.elo

{'SAS': 1500,
 'GSW': 1500,
 'LAL': 1500,
 'TOR': 1500,
 'IND': 1500,
 'ORL': 1500,
 'BKN': 1500,
 'CLE': 1500,
 'MEM': 1500,
 'NOP': 1500,
 'DEN': 1500,
 'MIA': 1508.2900264265638,
 'UTA': 1491.7099735734362,
 'OKC': 1500,
 'CHA': 1500,
 'ATL': 1500,
 'BOS': 1500,
 'MIN': 1500,
 'CHI': 1500,
 'PHX': 1500,
 'LAC': 1500,
 'PHI': 1500,
 'WAS': 1500,
 'MIL': 1500,
 'HOU': 1500,
 'DAL': 1500,
 'NYK': 1500,
 'DET': 1500,
 'SAC': 1500,
 'POR': 1500}

In [528]:
test.win_prob('MIA', 'UTA')

(0.6084568378350207, 0.3915431621649793)

In [529]:
test.expected_value('MIA', 'UTA',100, -111)

(0.21691367567004138, -0.25571525029900327, 1.0, 0.900900900900901)

In [530]:
test.Kelly_fraction('MIA', 'UTA',100, -111)

(0.025, 0.0)

In [531]:
test.simulation('MIA', 'UTA', 100, -111, 0)

(975.0, -25.0)

In [532]:
df = pd.read_csv("nba_2008-2025.csv")
df.head()

,season,date,regular,playoffs,away,home,score_away,score_home,q1_away,q2_away,...,ot_home,whos_favored,spread,total,moneyline_away,moneyline_home,h2_spread,h2_total,id_spread,id_total
0,2008,10/30/2007,True,False,por,sas,97,106,26,23,...,0,home,13.0,189.5,900.0,-1400.0,5.0,95.0,0.0,1
1,2008,10/30/2007,True,False,uta,gsw,117,96,28,34,...,0,home,1.0,212.0,100.0,-120.0,3.0,105.5,0.0,1
2,2008,10/30/2007,True,False,hou,lal,95,93,16,27,...,0,away,5.0,199.0,-230.0,190.0,3.0,99.0,0.0,0
3,2008,10/31/2007,True,False,phi,tor,97,106,22,28,...,0,home,6.5,191.0,255.0,-305.0,2.0,96.5,1.0,1
4,2008,10/31/2007,True,False,was,ind,110,119,23,22,...,16,away,1.5,203.5,-125.0,105.0,1.0,105.0,0.0,1


In [533]:
data = df[[	'season','date','away','home','score_away','score_home','moneyline_away','moneyline_home']]
data['home_win'] = (data['score_home'] > data['score_away']).astype(int)
data['date'] = pd.to_datetime(data.date)
data['away'] = [x.upper() for x in data.away]
data['home'] = [x.upper() for x in data.home]
data['score_home'] = data['score_home'].astype(int)
data['score_away'] = data['score_away'].astype(int)
data2022 = data[data['season'] == 2022]

C:\Users\JacobPolomsky\AppData\Local\Temp\ipykernel_14860\3898106702.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['home_win'] = (data['score_home'] > data['score_away']).astype(int)
C:\Users\JacobPolomsky\AppData\Local\Temp\ipykernel_14860\3898106702.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['date'] = pd.to_datetime(data.date)
C:\Users\JacobPolomsky\AppData\Local\Temp\ipykernel_14860\3898106702.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a 

In [534]:
data2022.head()

,season,date,away,home,score_away,score_home,moneyline_away,moneyline_home,home_win
17835,2022,2021-10-19,BKN,MIL,104,127,105.0,-125.0,1
17836,2022,2021-10-19,GSW,LAL,121,114,140.0,-160.0,0
17837,2022,2021-10-20,IND,CHA,122,123,-125.0,105.0,1
17838,2022,2021-10-20,CHI,DET,94,88,-220.0,190.0,0
17839,2022,2021-10-20,WAS,TOR,98,83,120.0,-140.0,0


In [535]:
data_sim = data2022[['home','away','moneyline_home','moneyline_away','home_win']]
game_profit = []
game_cash = []

data_sim

,home,away,moneyline_home,moneyline_away,home_win
17835,MIL,BKN,-125.0,105.0,1
17836,LAL,GSW,-160.0,140.0,0
17837,CHA,IND,105.0,-125.0,1
17838,DET,CHI,190.0,-220.0,0
17839,TOR,WAS,-140.0,120.0,0
...,...,...,...,...,...
19153,GSW,BOS,-200.0,175.0,1
19154,BOS,GSW,-155.0,135.0,1
19155,BOS,GSW,-165.0,145.0,0
19156,GSW,BOS,-165.0,145.0,1


In [536]:
data_sim = data2022[['home','away','moneyline_home','moneyline_away','home_win']]
game_profit = []
game_cash = []
for h_team, a_team, h_line, a_line, h_win in data_sim.iloc:
    cash, profit = test.simulation(h_team, a_team, h_line, a_line, h_win, kelly_limit=4)
    game_profit.append(profit)
    game_cash.append(cash)
    test.update_rating(h_team, a_team, h_win)


In [537]:
data2022['profit'] = game_profit
data2022['total_cash'] = game_cash
data2022

C:\Users\JacobPolomsky\AppData\Local\Temp\ipykernel_14860\3797047176.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data2022['profit'] = game_profit
C:\Users\JacobPolomsky\AppData\Local\Temp\ipykernel_14860\3797047176.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data2022['total_cash'] = game_cash


,season,date,away,home,score_away,score_home,moneyline_away,moneyline_home,home_win,profit,total_cash
17835,2022,2021-10-19,BKN,MIL,104,127,105.0,-125.0,1,9.750000,984.750000
17836,2022,2021-10-19,GSW,LAL,121,114,140.0,-160.0,0,0.000000,984.750000
17837,2022,2021-10-20,IND,CHA,122,123,-125.0,105.0,1,12.924844,997.674844
17838,2022,2021-10-20,CHI,DET,94,88,-220.0,190.0,0,-12.470936,985.203908
17839,2022,2021-10-20,WAS,TOR,98,83,120.0,-140.0,0,-1.279984,983.923924
...,...,...,...,...,...,...,...,...,...,...,...
19153,2022,2022-06-05,BOS,GSW,88,107,175.0,-200.0,1,-7.485365,591.343832
19154,2022,2022-06-08,GSW,BOS,100,116,135.0,-155.0,1,3.566118,594.909951
19155,2022,2022-06-10,GSW,BOS,107,97,145.0,-165.0,0,-7.436374,587.473576
19156,2022,2022-06-13,BOS,GSW,94,104,145.0,-165.0,1,-7.259909,580.213667


In [539]:
test.elo

{'SAS': np.float64(1459.675531795356),
 'GSW': np.float64(1665.1116321865197),
 'LAL': np.float64(1411.6154860448721),
 'TOR': np.float64(1578.1095501102102),
 'IND': np.float64(1337.4127965504965),
 'ORL': np.float64(1365.6209133082264),
 'BKN': np.float64(1492.490455680649),
 'CLE': np.float64(1468.9143396643185),
 'MEM': np.float64(1615.9330154666939),
 'NOP': np.float64(1514.4525815041536),
 'DEN': np.float64(1532.7738395258646),
 'MIA': np.float64(1626.1323908670145),
 'UTA': np.float64(1530.8518918504692),
 'OKC': np.float64(1347.5702943976169),
 'CHA': np.float64(1517.274134848823),
 'ATL': np.float64(1538.2219972763526),
 'BOS': np.float64(1640.5474461313756),
 'MIN': np.float64(1562.4841823189654),
 'CHI': np.float64(1490.008946727366),
 'PHX': np.float64(1647.7626389985621),
 'LAC': np.float64(1506.8616104063688),
 'PHI': np.float64(1595.7433426252962),
 'WAS': np.float64(1414.690192761535),
 'MIL': np.float64(1608.6413814507137),
 'HOU': np.float64(1311.610089508821),
 'DAL'

In [540]:
data2022[data2022.profit >= 0].profit.sum()

np.float64(6795.2907190725)

In [541]:
data2022[data2022.profit <= 0].profit.sum()

np.float64(-7181.18547516638)

In [542]:
data2022.to_csv("data2022v2static100")

### Betting

In [543]:
def brier_baseline(data):
    baseline = data.home_win.mean()
    brier = np.mean((baseline - data.home_win) ** 2)
    return brier

In [544]:
def brier_model(data):
    prediction = data.h_prob
    brier_score = np.mean((prediction - data.home_win) ** 2)
    return brier_score

In [545]:
def price_to_prob(price):
    
    if price > 0:
        p_hat = 100 / (price + 100)
    else:
        p_hat = -price / (-price + 100)
    return p_hat

In [546]:
def implied_prob(ph_home, ph_away):
    s = ph_home + ph_away
    p_home = ph_home / s
    p_away = ph_away / s
    return [p_home, p_away]